# Expérience 3 — ALS (bibliothèque `implicit`)

**Question centrale** : faut-il entraîner l'ALS sur toute la période
d'entraînement, ou seulement sur les dernières heures ?

Les artefacts du dépôt utilisent l'intégralité des clics. Pour un flux
d'actualité, ce choix n'est pas évident : les co-lectures d'il y a dix jours
portent-elles encore de l'information ?

Second réglage : le **nombre de facteurs latents**.

⚠️ Ce notebook ré-entraîne plusieurs modèles ALS — compter environ une minute par
configuration.

## Protocole commun

Identique dans tous les notebooks d'expérimentation, sinon les chiffres ne sont pas
comparables :

- **découpage temporel 60 / 20 / 20** sur `click_timestamp` ;
- artefacts construits sur la **seule** période d'entraînement (`models_split/`) ;
- réglage sur la **validation** ; la période de test reste intacte jusqu'à la mesure
  finale (notebook 07) ;
- lecteurs évalués : connus à l'entraînement **et** actifs pendant la période
  d'évaluation ;
- métriques : HitRate@5, Recall@5, couverture, personnalisation.

> Prérequis : `python -m src.evaluate --data-dir data/news-portal-user --out-dir models_split`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('..')

from src import experiments as xp
from src.recommender import Recommender

DATA = Path('..') / 'data' / 'news-portal-user'
MODELS = Path('..') / 'models_split'

train, val, test = xp.load_split(DATA)
reco = Recommender(MODELS)
users, cible = xp.eval_users(reco, val, max_users=2000)
print(f'{len(users):,} lecteurs évalués sur la période de validation')

[clicks] 1 fichier(s) vide(s) ignoré(s) : clicks_hour_100.csv


[split] entraînement 1,792,908 | validation 597,636 | test 597,637
[split] bornes temporelles : t60=1507602953792 t80=1507843212579


2,000 lecteurs évalués sur la période de validation


## 1. Fenêtre d'entraînement

L'évaluation se fait toujours sur le même vivier récent, pour que seule la fenêtre
d'apprentissage varie.

In [2]:
# Vivier de 6 h : l'optimum du contenu (notebook 04). Un vivier plus court
# avantagerait mécaniquement la popularité et fausserait la comparaison.
POOL_HEURES = 6
pool = xp.recent_pool(train, POOL_HEURES)

configs = {}
for heures in (None, 240, 72, 24):
    fabrique = xp.train_als_window(train, heures, factors=50)
    nom = 'toute la période' if heures is None else f'{heures} dernières h'
    configs[nom] = fabrique(pool, reco)

xp.compare(configs, users, cible, n_articles=reco.n_articles)

  0%|          | 0/15 [00:00<?, ?it/s]

[als] fenêtre complète h : 255,516 users x 29,118 items, 1,792,908 clics


  0%|          | 0/15 [00:00<?, ?it/s]

[als] fenêtre 240 h : 255,516 users x 29,118 items, 1,792,908 clics


  0%|          | 0/15 [00:00<?, ?it/s]

[als] fenêtre 72 h : 119,937 users x 11,244 items, 500,314 clics


  0%|          | 0/15 [00:00<?, ?it/s]

[als] fenêtre 24 h : 72,055 users x 6,777 items, 256,329 clics


,HitRate@5,Recall@5,couverture %,personnalisation %
configuration,,,,
toute la période,0.0120,0.0016,0.031,84.3
240 dernières h,0.0120,0.0016,0.031,84.3
72 dernières h,0.0200,0.0024,0.042,97.4
24 dernières h,0.0175,0.0021,0.042,100.0


## 2. Nombre de facteurs latents

À fenêtre fixée, on fait varier la dimension du modèle.

In [3]:
MEILLEURE_FENETRE = 24     # ajuster selon le tableau précédent

configs = {}
for facteurs in (16, 50, 100):
    fabrique = xp.train_als_window(train, MEILLEURE_FENETRE, factors=facteurs)
    configs[f'{facteurs} facteurs'] = fabrique(pool, reco)

xp.compare(configs, users, cible, n_articles=reco.n_articles)

  0%|          | 0/15 [00:00<?, ?it/s]

[als] fenêtre 24 h : 72,055 users x 6,777 items, 256,329 clics


  0%|          | 0/15 [00:00<?, ?it/s]

[als] fenêtre 24 h : 72,055 users x 6,777 items, 256,329 clics


  0%|          | 0/15 [00:00<?, ?it/s]

[als] fenêtre 24 h : 72,055 users x 6,777 items, 256,329 clics


,HitRate@5,Recall@5,couverture %,personnalisation %
configuration,,,,
16 facteurs,0.0345,0.0039,0.018,100.0
50 facteurs,0.0175,0.0021,0.042,100.0
100 facteurs,0.0125,0.0011,0.071,100.0


## Lecture

À retenir : un modèle collaboratif a besoin de **co-lectures**, et un article
publié il y a deux heures n'en a presque aucune. C'est une limite structurelle, pas
un défaut de réglage — réduire la fenêtre d'entraînement ne la contourne pas, elle
prive au contraire le modèle des données dont il a besoin.

L'ALS conserve un avantage réel sur la popularité : une personnalisation élevée.
Il propose des articles différents à des lecteurs différents, ce que la popularité
ne fait jamais.